# E-Commerce Delivery Delay & Supply Chain Risk Predictor
### End-to-End Analytics, Leak-Free Feature Engineering & MLOps Benchmark
**Author:** Senior Supply Chain Analytics Engineer & Data Scientist  
**Dataset:** Olist Brazilian E-Commerce Public Dataset  
**Target Roles:** Amazon, Flipkart, Myntra

## 1. Environment & Module Setup

In [ ]:
import sys
from pathlib import Path

# Ensure project root in sys.path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from src.db_ingest import get_db_connection, DEFAULT_DB_PATH
from src.queries import load_analytical_base_table, load_state_kpis, load_category_kpis
from src.features import engineer_features, FEATURE_COLUMNS, LEAKY_COLUMNS
from src.evaluate import evaluate_predictions, optimize_classification_threshold

sns.set_theme(style="whitegrid")
print("All modules imported successfully.")

## 2. Load Analytical Base Table from SQLite DW
We query the order-level analytical base table constructed with advanced CTEs and window functions.

In [ ]:
conn = get_db_connection(DEFAULT_DB_PATH)
abt_df = load_analytical_base_table(conn)
state_kpis = load_state_kpis(conn)
conn.close()

print(f"Loaded {len(abt_df):,} delivered order records.")
print(f"Overall empirical delay rate: {abt_df['is_delayed'].mean():.2%}")
abt_df.head()

## 3. Exploratory Data Analysis & Supply Chain Bottlenecks

In [ ]:
# State-level delay rate leaderboard
plt.figure(figsize=(10, 5))
sns.barplot(data=state_kpis.sort_values(by='delay_rate_pct', ascending=False), x='customer_state', y='delay_rate_pct', palette='Reds_r')
plt.title("Delay Rate (%) by Destination State")
plt.ylabel("Delay Rate (%)")
plt.xlabel("Customer State")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 4. Leak-Free Feature Engineering Inspection
Verify that no post-delivery survey scores or actual customer delivery timestamps are leaked into the predictor set.

In [ ]:
# Assert zero overlap between feature set and leaky variables
overlap = set(FEATURE_COLUMNS).intersection(set(LEAKY_COLUMNS))
assert len(overlap) == 0, f"Leakage detected! {overlap}"
print("Zero leakage audit passed: All 26 candidate predictors available at or before dispatch.")

feat_df, seller_summary = engineer_features(abt_df)
feat_df[['distance_km', 'dispatch_time_hours', 'seller_historical_delay_rate', 'freight_to_price_ratio', 'weight_density']].describe()

## 5. Model Benchmarking & Performance Leaderboard

In [ ]:
pipeline_pkg = joblib.load(Path("../models/calibrated_pipeline.joblib"))
model_card_path = Path("../models/model_card.json")
import json
with open(model_card_path) as f:
    card = json.load(f)

leaderboard = pd.DataFrame(card["benchmark_leaderboard"])
print("Candidate Model Leaderboard on Holdout Validation Set:")
leaderboard[['model_name', 'pr_auc', 'roc_auc', 'brier_score', 'f1_score']]

## 6. Calibration & Operational Threshold Optimization

In [ ]:
print(f"Winning Architecture: {card['best_model_name']}")
print(f"Calibrated Optimal Decision Threshold: {card['best_threshold']:.3f}")
print("Test Set Performance at Tuned Threshold:")
for k, v in card["test_metrics_tuned"].items():
    print(f"  {k:<20}: {v}")

## 7. Global Feature Importance (SHAP Explainability)

In [ ]:
shap_drivers = pd.Series(card["top_shap_features"]).sort_values(ascending=True)
plt.figure(figsize=(9, 5))
shap_drivers.plot(kind='barh', color='#1f77b4')
plt.title("Top Root Causes of Delivery Delay (Global Mean |SHAP|)")
plt.xlabel("Mean |SHAP Value|")
plt.tight_layout()
plt.show()